In [2]:
%load_ext autoreload
%autoreload 2

import networkx as nx
from pathlib import Path
import pipeline
import pandas as pd
from collections import defaultdict
from geopy.distance import distance

### build initial full network (if needed)

In [ ]:
data = Path('data/undir_trials/24h/depnet_undirected_24h.gz')
df = pd.read_csv(data)
df['DEP'] = pd.to_numeric(df['DEP'], errors='coerce')
df = df.dropna(subset=['DEP'])

filepath = f'data/undir_trials/24h/full_network_24h.txt'
df.to_csv(filepath, sep=' ', index=False, header=False)

print(f"Saved with {len(df)} edges.")

load in

In [4]:
gpath = Path('data/undir_trials/24h/full_network_24h.txt')
G = pipeline.load(gpath)

# --- add census tract attribute ---
pipeline.node_to_area(G)
num_empty = sum(1 for n, d in G.nodes(data=True) if d.get('tract') == 'Unknown')
print(f'% empty: {((num_empty/len(G.nodes()))*100):.4f}')

Nodes: 115972
Edges: 15133742
% empty: 9.3204


### map POIs to buckets (creating skeleton)

In [5]:
# --- create blank network ---

# collect cats and tracts
cats = set(nx.get_node_attributes(G, 'poi_type').values())
tracts = set(nx.get_node_attributes(G, 'tract').values())
agg_nodes = {f"{c}||{t}" for c in cats for t in tracts}

print(f'len cats: {len(cats)} -- len tracts: {len(tracts)} -- # combos: {len(agg_nodes)}')

# init network
G_agg = nx.Graph()
G_agg.add_nodes_from(agg_nodes)

# --- map nodes to buckets ---

mapping = defaultdict(list)

for poi, attrs in G.nodes(data=True):
    c = attrs.get('poi_type')
    t = attrs.get('tract')
    key = f"{c}||{t}"

    if key in G_agg:
        mapping[key].append(poi)

# prune empty nodes
empty_nodes = [node for node in G_agg.nodes() if node not in mapping]
G_agg.remove_nodes_from(empty_nodes)
len(G_agg.nodes())

len cats: 20 -- len tracts: 1029 -- # combos: 20580


15025

### edges and node attrs

In [8]:
import pickle

# sum up visits (node attr)
for bucket, poi_list in mapping.items():
    total_visits = sum(G.nodes[poi].get('total_visits', 0) for poi in poi_list)
    G_agg.nodes[bucket]['total_visits'] = total_visits

# --- add edges ---

# create fast lookup
poi_to_bucket = {}
for bucket, poi_list in mapping.items():
    for poi in poi_list:
        poi_to_bucket[poi] = bucket

# add based on lookup (incl attrs)
for u, v, attrs in G.edges(data=True):
    bucket_u = poi_to_bucket.get(u)
    bucket_v = poi_to_bucket.get(v)

    if bucket_u and bucket_v and (bucket_u != bucket_v):
        cur_weight = 0
        cur_cov = 0
        new_cov = attrs.get('N_COVISITS', 0)
        # check if already exists
        if G_agg.has_edge(bucket_u, bucket_v):
            cur_weight = G_agg[bucket_u][bucket_v].get('weight', 0)
            cur_cov = G_agg[bucket_u][bucket_v].get('N_COVISITS', 0)
        # upsert (create or override)
        G_agg.add_edge(bucket_u, bucket_v, weight=cur_weight+1, N_COVISITS=cur_cov+new_cov)

# --- recompute dep ---

for i, j, attrs in G_agg.edges(data=True):
    cv = attrs.get('N_COVISITS')
    visits_i = G_agg.nodes[i]['total_visits']
    visits_j = G_agg.nodes[j]['total_visits']

    attrs['DEP'] = cv / (visits_i * visits_j)

# --- lat/lon, distance, category ---

# gather internal points of census tracts
gaz = pd.read_csv('data/geo/2020_gaz_tracts_25.txt', sep='\t', dtype={'GEOID': str})
gaz.columns = gaz.columns.str.strip()
tract_coords = gaz.set_index('GEOID')[['INTPTLAT', 'INTPTLONG']].to_dict('index')

# nodes (poi_type + lat/lon)
for node, attrs in G_agg.nodes(data=True):
    c, t = node.split('||')
    attrs['poi_type'] = c

    if t in tract_coords:
        attrs['latitude'] = float(tract_coords[t]['INTPTLAT'])
        attrs['longitude'] = float(tract_coords[t]['INTPTLONG'])
    else:
        attrs['latitude'] = None
        attrs['longitude'] = None

# edges (distance)
for i, j, attrs in G_agg.edges(data=True):
    lat1, lon1 = G_agg.nodes[i].get('latitude'), G_agg.nodes[i].get('longitude')
    lat2, lon2 = G_agg.nodes[j].get('latitude'), G_agg.nodes[j].get('longitude')

    if None not in (lat1, lon1, lat2, lon2):
        attrs['DIST_KM'] = distance((lat1, lon1), (lat2, lon2)).km
    else:
        attrs['DIST_KM'] = None

# save
# with open('data/agg/agg_network_24h_undir.pkl', 'wb') as f:
#     pickle.dump(G_agg, f)
# print(f'Saved with {len(G_agg.nodes())} nodes and {len(G_agg.edges())} edges')

In [10]:

# DIAGNOSTICS: bucket composition
print(f'=== BUCKET DIAGNOSTICS ===')
print(f'Categories sample: {sorted(cats)[:10]}')
print(f'Tracts sample: {sorted(tracts)[:10]}')
print(f'Theoretical max possible edges (full complete graph): {len(G_agg.nodes()) * (len(G_agg.nodes()) - 1) // 2:,}')

# Unmapped POIs (excluded from aggregation)
unmapped_pois = [n for n in G.nodes() if n not in poi_to_bucket]
print(f'Unmapped POIs (no category/tract): {len(unmapped_pois)}')

# Bucket occupancy
bucket_sizes = {k: len(v) for k, v in mapping.items()}
print(f'Non-empty buckets: {len(bucket_sizes)}')
print(f'Max POIs in single bucket: {max(bucket_sizes.values())}')
print(f'Min POIs in single bucket: {min(bucket_sizes.values())}')

# Distribution
print(f'Buckets with 1 POI: {sum(1 for s in bucket_sizes.values() if s == 1)}')
print(f'Buckets with 10+ POIs: {sum(1 for s in bucket_sizes.values() if s >= 10)}')
print(f'Total POIs mapped: {sum(bucket_sizes.values())}')


=== BUCKET DIAGNOSTICS ===
Categories sample: ['Agriculture', 'Arts / Museum', 'Automotive Service', 'City / Outdoors', 'Coffee / Tea', 'College', 'Entertainment', 'Financial Service', 'Food', 'Grocery']
Tracts sample: ['25009201100', '25009202101', '25009202103', '25009202104', '25009202200', '25009203100', '25009203200', '25009203301', '25009203302', '25009204101']
Theoretical max possible edges (full complete graph): 112,867,800
Unmapped POIs (no category/tract): 0
Non-empty buckets: 15025
Max POIs in single bucket: 2055
Min POIs in single bucket: 1
Buckets with 1 POI: 3096
Buckets with 10+ POIs: 3160
Total POIs mapped: 115972


In [8]:
# confirm input file used
print("file size / lines:", Path('data/undir_trials/24h/full_network_24h.txt').stat().st_size)

# compare counts
print(len(G_agg.nodes()), len(G_agg.edges()))

# are aggregated edges unique or multiedges?
print("Graph type:", type(G_agg))
print("sum of weights:", sum(d.get('weight',1) for _,_,d in G_agg.edges(data=True)))
# unique bucket pairs implied by raw edges:
pairs = set()
for u,v,_ in G.edges(data=True):
    bu, bv = poi_to_bucket.get(u), poi_to_bucket.get(v)
    if bu and bv and bu!=bv:
        pairs.add(tuple(sorted((bu,bv))))
print(len(pairs), len(G.edges()), sum(1 for e in G.edges() if poi_to_bucket.get(e[0]) and poi_to_bucket.get(e[1]) and poi_to_bucket.get(e[0])!=poi_to_bucket.get(e[1])))
# inspect mapping sizes
max((len(v) for v in mapping.values()))

file size / lines: 2129673086
15025 5239856
Graph type: <class 'networkx.classes.graph.Graph'>
sum of weights: 14764232
5239856 15133742 14764232


2055